In [1]:
%load_ext autoreload
%autoreload 2

# I. Modeling

Since our model is designed to encode the attributes of each entity (**type, size, color, and orientation angle**) we adopt a **multi-task CNN encoder architecture**. 

The encoder is jointly trained to predict these attributes from the visual input (crops) and to learn a compact **latent representation**. 

This latent vector will be combined with the entity’s **spatial and structural information** (**problem ID, panel ID, and bounding boxes**).

The resulting feature representation is then provided to the **graph construction module**, which uses these attributes to construct the graph representation that will ultimately be processed by the **GNN**.



## 1. Visual Encoder (CNN)

we choose to use a multi-task CNN architecture 
<div align="center">
  <img src="../assets/images/MT-CNN_GB.drawio.png" alt="Alt text">
</div>

**Getting the data**

In [ ]:
# test the output shape of the model
# we need first to grab the data 

from hri_gnn.datasets import RavenDataset , gnn_flat_collate
from torch.utils.data import DataLoader

# Train dataset
train_dataset = RavenDataset(
    dataset="raven-20000",
    split="train",
    percentage_limit=0.01
)
# Test dataset
test_dataset = RavenDataset(
    dataset="raven-20000",
    split="test",
    percentage_limit=0.01
)
# Validation dataset
val_dataset = RavenDataset(
    dataset="raven-20000",
    split="val",
    percentage_limit=0.01
)
# Train DataLoader
train_datatloader = DataLoader(
    dataset=train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn= gnn_flat_collate
)
# Test DataLoader
test_datatloader = DataLoader(
    dataset=train_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn= gnn_flat_collate
)
# Validation DataLoader
val_datatloader = DataLoader(
    dataset=train_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn= gnn_flat_collate
)


In [ ]:
# Checking datasets lengths 
len(train_dataset) ,len(val_dataset), len(test_dataset)

(814, 271, 271)

In [ ]:
# Instanciating the model

import torch
from hri_gnn.models.cnn_encoder import RavenCnnEncoder

# Fixing the seeds
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# make the code device agnostic 
device = "cuda" if torch.cuda.is_available() else "cpu"
# printing the device
print(f"the available device is:{device}")

cnn_encoder = RavenCnnEncoder().to(device)

the available device is:cuda
the model device is: cuda:0


**Training the model**

In [ ]:
import torch 
from hri_gnn.training.trainer import Trainer
from hri_gnn.losses import CompositeLoss
from hri_gnn.metrics import TrainingTimer

criterion = CompositeLoss()
optimizer = torch.optim.Adam(params=cnn_encoder.parameters(),lr=0.01)
with TrainingTimer(model= cnn_encoder) as train_timer:
    raven_trainer = Trainer(
                        model=cnn_encoder, 
                        optimizer=optimizer, 
                        criterion=criterion)
    taining_output = raven_trainer.fit(train_datatloader,test_datatloader,epochs=3)
model_training_time = train_timer.elapsed


start training model of type: 'RavenCnnEncoder' on device: cuda:0 ....
No device provided, Trainer will use: cuda
No device was specified, CompositeMetric is setting device to: cuda


  0%|          | 0/3 [00:00<?, ?it/s]

## 2. Reasoning System (Graph Builder, GNN & HRI) 